[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/07_ml_interview_drills/07_ml_interview_drills.ipynb)

# 07 · ML 面试 Drills（capstone）

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
把前六章的能力组织成面试级别的推导与验证：softmax-CE 梯度、k-fold CV、bootstrap 误差棒——全部在真实 **Wine** 多分类数据上。

**你将完成：**
1. 数值稳定的 softmax + 交叉熵
2. softmax-CE 梯度 = `p - onehot`（推导 + 数值验证）
3. k-fold 交叉验证选超参（L2 强度）
4. bootstrap 置信区间（给准确率配误差棒）

> 数据：UCI Wine（178 瓶, 13 化学特征, 3 个品种）。

## 0 · 加载真实数据（Wine 三分类）

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
CACHE=os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE,exist_ok=True)
def fetch(u,f):
    p=os.path.join(CACHE,f)
    if not os.path.exists(p): urllib.request.urlretrieve(u,p)
    return p
wine=pd.read_csv(fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data","wine.data"),header=None)
X=wine.iloc[:,1:].to_numpy(float); y=wine.iloc[:,0].to_numpy(int)-1   # 类别 0,1,2
X=(X-X.mean(0))/X.std(0)
K=3
print(f"X:{X.shape}  类别:{np.bincount(y)}")

## 1 · 数值稳定的 softmax + 交叉熵

softmax 必须先减 max 防 overflow。交叉熵 = -log p[真实类]。

In [ ]:
def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)         # 数值稳定
    e = np.exp(Z); return e/e.sum(1, keepdims=True)
def cross_entropy(P, y):
    return -np.log(P[np.arange(len(y)), y] + 1e-12).mean()

# 验证数值稳定：巨大 logit 不爆
Zbig = np.array([[1000., 1001., 999.]])
print("大 logit softmax:", softmax(Zbig), "(没 overflow ✓)")
print("均匀 logit 的 CE =", round(cross_entropy(softmax(np.zeros((1,K))), np.array([0])),4),
      "= log(3) =", round(np.log(3),4))

## 2 · softmax-CE 梯度 = p − onehot（数值验证）

课上推导：`∂L/∂z = p - y_onehot`。这里训练一个多分类逻辑回归并数值校验这个梯度。

In [ ]:
def onehot(y, K):
    O=np.zeros((len(y),K)); O[np.arange(len(y)),y]=1; return O

def grad_W(W, X, y):
    P = softmax(X@W)
    return X.T @ (P - onehot(y, K)) / len(y)    # (d,K)

rng=np.random.default_rng(0); W=rng.normal(0,0.1,(X.shape[1],K))
ana = grad_W(W, X, y)
# 数值校验若干元素
eps=1e-5; ok=True
for (i,j) in [(0,0),(3,1),(7,2),(12,0)]:
    Wp=W.copy(); Wp[i,j]+=eps; Wm=W.copy(); Wm[i,j]-=eps
    num=(cross_entropy(softmax(X@Wp),y)-cross_entropy(softmax(X@Wm),y))/(2*eps)
    ok &= abs(num-ana[i,j])<1e-4
print("softmax-CE 梯度数值校验:", "通过 ✓" if ok else "失败 ✗")

## 3 · k-fold 交叉验证选 L2

用 5-fold CV 在训练数据内部选最优 L2 强度，不碰测试集。

In [ ]:
def fit_softmax(X, y, l2=0.0, lr=0.5, steps=300):
    W=np.zeros((X.shape[1],K))
    for _ in range(steps):
        P=softmax(X@W); W-=lr*(X.T@(P-onehot(y,K))/len(y)+l2*W)
    return W
def acc(W,X,y): return (softmax(X@W).argmax(1)==y).mean()

def kfold_score(X, y, l2, k=5, seed=0):
    rng=np.random.default_rng(seed); idx=rng.permutation(len(y))
    folds=np.array_split(idx, k); scores=[]
    for i in range(k):
        val=folds[i]; trn=np.concatenate([folds[j] for j in range(k) if j!=i])
        W=fit_softmax(X[trn],y[trn],l2=l2)
        scores.append(acc(W,X[val],y[val]))
    return np.mean(scores), np.std(scores)

for l2 in [0.0, 0.01, 0.1, 1.0]:
    m,s=kfold_score(X,y,l2)
    print(f"L2={l2:<5} CV acc = {m:.3f} ± {s:.3f}")

## 4 · bootstrap 置信区间

给测试准确率配 95% CI：有放回重采样测试集 B 次，取分位数。

In [ ]:
rng=np.random.default_rng(1); idx=rng.permutation(len(y)); cut=int(0.7*len(idx))
tr,te=idx[:cut],idx[cut:]
W=fit_softmax(X[tr],y[tr],l2=0.01)
preds=(softmax(X[te]@W).argmax(1)==y[te]).astype(float)
point=preds.mean()

B=2000; boots=np.array([preds[rng.integers(0,len(preds),len(preds))].mean() for _ in range(B)])
lo,hi=np.percentile(boots,[2.5,97.5])
print(f"test accuracy = {point:.3f}   95% bootstrap CI = [{lo:.3f}, {hi:.3f}]")
print(f"=> 在 n={len(te)} 的测试集上，准确率的不确定度约 ±{(hi-lo)/2:.3f}")

## 5 · 经典概率题：贝叶斯定理与基率谬误

面试里几乎必出一道概率题，最经典的就是医学检测：**某病患病率 1%，检测敏感度 99%、特异度 95%；某人检测阳性，他真患病的概率是多少？** 凭直觉很多人脱口而出「95% 左右」，但正确答案约 **16.7%（1/6）**。下面用贝叶斯公式算一遍，再用「每 10 万人按比例计数」的频率自然法验证两者一致，最后展示后验对**基率（先验患病率）**有多敏感——这就是「基率谬误」。它和评测的联系很直接：稀有事件（低基率）的检测器，看似漂亮的准确率会严重误导，必须看 precision/PR-AUC。

In [ ]:
# 经典面试概率题（基率谬误）：患病率 1%、检测敏感度 99%、特异度 95%。
# 阳性后真患病的概率 P(D|+) 是多少？凭直觉很多人答 ~95%，正确答案约 16.7%。
def posterior_disease(prevalence, sensitivity, specificity):
    # Bayes: P(D|+) = P(+|D)P(D) / [P(+|D)P(D) + P(+|¬D)P(¬D)]
    p_pos_given_d  = sensitivity                 # 真阳性率
    p_pos_given_nd = 1 - specificity             # 假阳性率
    num = p_pos_given_d * prevalence
    den = num + p_pos_given_nd * (1 - prevalence)
    return num / den

# 用频率自然法验证（每 100000 人按比例算期望计数，不用随机）
N = 100_000; prev, sens, spec = 0.01, 0.99, 0.95
diseased = N * prev; healthy = N * (1 - prev)
TP = diseased * sens; FN = diseased * (1 - sens)
TN = healthy * spec;  FP = healthy * (1 - spec)
post_counts = TP / (TP + FP)
post_bayes = posterior_disease(prev, sens, spec)
print(f"贝叶斯公式  P(患病|阳性) = {post_bayes:.4f}  ≈ {post_bayes:.1%}")
print(f"频率计数法  TP/(TP+FP) = {TP:.0f}/({TP:.0f}+{FP:.0f}) = {post_counts:.4f}  （两法一致 ✓）")
print("=> 直觉陷阱：检测很准(99/95)，但因患病率只有 1%，大量假阳性来自庞大的健康人群")
assert abs(post_bayes - post_counts) < 1e-9, "Bayes 公式与频率计数应一致"
assert abs(post_bayes - 1/6) < 0.02, "标准题答案约 1/6 ≈ 16.7%"

# 后验对基率(先验)极其敏感：同样的检测，患病率不同时后验天差地别
for prev in [0.001, 0.01, 0.10, 0.50]:
    print(f"  患病率={prev:6.1%} -> P(患病|阳性)={posterior_disease(prev, sens, spec):.3f}")
print("=> 这就是『基率谬误』：忽视先验(患病率)会把阳性结果严重高估；评测里同理——稀有事件的检测器要看 PR 而非看似漂亮的准确率")

---
## ✏️ 练习区

### ✏️ 练习 1：数值稳定 softmax + 交叉熵

实现 `softmax_stable(Z)`（减 max）和 `ce_loss(Z, y)`（直接从 logit 算，用 logsumexp 风格避免两次指数）。

In [ ]:
def softmax_stable(Z):
    # TODO: 减每行 max 再 exp 归一化
    raise NotImplementedError
def ce_loss(Z, y):
    # TODO: 平均交叉熵；可用 logsumexp: L = mean(logsumexp(Z) - Z[i,y])
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
Z=np.array([[1000.,1001.,999.],[0.,0.,0.]])
P=softmax_stable(Z)
assert np.isfinite(P).all() and np.allclose(P.sum(1),1)
assert np.allclose(P[1],[1/3,1/3,1/3])
assert abs(ce_loss(np.zeros((1,3)), np.array([0])) - np.log(3)) < 1e-9
# ce_loss 应等于 -log(softmax)[真实类]
assert abs(ce_loss(Z, np.array([1,0])) - (-np.log(softmax_stable(Z)[[0,1],[1,0]])).mean()) < 1e-9
print("练习 1 通过 ✓")


### ✏️ 练习 2：softmax-CE 梯度

实现 `softmax_ce_grad(Z, y)` 返回 `∂L/∂Z`（形状同 Z），即 `(softmax(Z) - onehot)/n`。

In [ ]:
def softmax_ce_grad(Z, y):
    # TODO: (softmax(Z) - onehot(y)) / n
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测（对 logit Z 数值校验）——
rng=np.random.default_rng(2); Z=rng.normal(size=(5,K)); yy=rng.integers(0,K,5)
ana=softmax_ce_grad(Z,yy); eps=1e-5; num=np.zeros_like(Z)
for i in range(5):
  for k in range(K):
    Zp=Z.copy();Zp[i,k]+=eps; Zm=Z.copy();Zm[i,k]-=eps
    num[i,k]=(ce_loss(Zp,yy)-ce_loss(Zm,yy))/(2*eps)
assert np.allclose(ana,num,atol=1e-5)
print("练习 2 通过 ✓  —— 数值梯度确认 ∂L/∂z = (p - onehot)/n")


### ✏️ 练习 3：k-fold CV

实现 `cv_select_l2(X, y, l2_grid, k)`：返回 CV 平均准确率最高的 L2。用提供的 `fit_softmax`/`acc`。

In [ ]:
def cv_select_l2(X, y, l2_grid, k=5, seed=0):
    # TODO: 对每个 l2 做 k-fold，返回平均 acc 最高的 l2
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
best=cv_select_l2(X, y, [0.0,0.01,0.1,1.0,10.0])
assert best in [0.0,0.01,0.1,1.0,10.0]
# 极端大正则(10)不该被选（欠拟合）
assert best != 10.0
print(f"练习 3 通过 ✓  CV 选出的最优 L2 = {best}")


### ✏️ 练习 4：bootstrap 置信区间

实现 `bootstrap_ci(correct, B, alpha, seed)`：`correct` 是 0/1 正确标记数组，
返回 `(point, lo, hi)` 的 (1-alpha) 置信区间。

In [ ]:
def bootstrap_ci(correct, B=2000, alpha=0.05, seed=0):
    # TODO: 有放回重采样 B 次算均值，取 [alpha/2, 1-alpha/2] 分位数
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
W=fit_softmax(X[tr],y[tr],l2=0.01)
correct=(softmax(X[te]@W).argmax(1)==y[te]).astype(float)
pt,lo,hi=bootstrap_ci(correct, B=2000)
assert lo <= pt <= hi and abs(pt-correct.mean())<1e-9
assert hi-lo > 0, "CI 应有宽度"
# 全对的话 CI 应贴近 1
pt2,lo2,hi2=bootstrap_ci(np.ones(50))
assert lo2==1.0 and hi2==1.0
print(f"练习 4 通过 ✓  acc={pt:.3f}  95% CI=[{lo:.3f},{hi:.3f}]")


---
## 📖 参考答案

In [ ]:
# 练习 1
def softmax_stable(Z):
    Z=Z-Z.max(1,keepdims=True); e=np.exp(Z); return e/e.sum(1,keepdims=True)
def ce_loss(Z, y):
    Zs=Z-Z.max(1,keepdims=True)
    lse=np.log(np.exp(Zs).sum(1))+Z.max(1)
    return float((lse - Z[np.arange(len(y)),y]).mean())
print("练习 1 ✓")

In [ ]:
# 练习 2
def softmax_ce_grad(Z, y):
    P=softmax_stable(Z); O=np.zeros_like(Z); O[np.arange(len(y)),y]=1
    return (P-O)/len(y)
print("练习 2 ✓")

In [ ]:
# 练习 3
def cv_select_l2(X, y, l2_grid, k=5, seed=0):
    best=(None,-1)
    for l2 in l2_grid:
        m,_=kfold_score(X,y,l2,k=k,seed=seed)
        if m>best[1]: best=(l2,m)
    return best[0]
print("练习 3 ✓")

In [ ]:
# 练习 4
def bootstrap_ci(correct, B=2000, alpha=0.05, seed=0):
    rng=np.random.default_rng(seed); n=len(correct)
    boots=np.array([correct[rng.integers(0,n,n)].mean() for _ in range(B)])
    lo,hi=np.percentile(boots,[100*alpha/2,100*(1-alpha/2)])
    return float(correct.mean()), float(lo), float(hi)
print("练习 4 ✓ —— 误差棒是评测科学家的肌肉记忆，C3 课会把它用到 LLM eval 上")